In [1]:
import sys
import importlib
import warnings
import random
from collections import defaultdict
import pickle
import os
import re

import torch
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


sys.path.append("A:/Masters/Thesis/Dr. Mohsen Rashwan/projects/")

from stt_benchmarking.utils import helpers, metrics as mt, validate
from stt_benchmarking.etl import load_english_data

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_colwidth", None)
random.seed(42)

print(device)

The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.
The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.


cuda


In [ ]:
data_path = "../../data/librispeech/LibriSpeech/test-clean"
records = load_english_data.load_librispeech(data_path)

In [2]:
def save_bootstrap_samples(samples, filepath="../../reports/phase_1_4/lists_info/librespeech/bootstrap_samples.pkl"):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'wb') as f:
        pickle.dump(samples, f)

def load_bootstrap_samples(filepath="../../reports/phase_1_4/lists_info/librespeech/bootstrap_samples.pkl"):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

In [4]:
save_bootstrap_samples(records, filepath="../../reports/phase_1_4/lists_info/librespeech/records.pkl")

In [3]:
records = load_bootstrap_samples("../../results/librespeech/audios/records.pkl")
len(records)

2620

In [4]:
filtered_records = [record for record in records if record["audio_duration"] <= 30]
len(filtered_records)

2611

In [5]:
records_bigger_30 = [record for record in records if record["audio_duration"] > 30]
len(records_bigger_30)

9

# Models

In [6]:
import json
import inspect
import os

def save_model_info(obj, attr_name="samples_info", save_dir="../../reports/phase_1_4/models_info/librespeech", is_llm=False, level=0):
    """
    Save the model's info dictionary to a JSON file.
    
    Parameters
    ----------
    obj : object
        Class object containing the info dictionary (e.g., seamless_m4t).
    attr_name : str, optional
        The attribute name holding the dictionary (default: "samples_info").
    save_dir : str, optional
        Path to directory where file will be saved.
    """
    # Ensure save directory exists
    os.makedirs(save_dir, exist_ok=True)

    # Get class object name
    obj_name = [name for name, val in inspect.currentframe().f_back.f_locals.items() if val is obj]
    obj_name = obj_name[0] if obj_name else obj.__class__.__name__

    # Extract info
    if not hasattr(obj, attr_name):
        raise AttributeError(f"Object has no attribute '{attr_name}'")
    
    info_dict = getattr(obj, attr_name)

    # Save as JSON
    if is_llm:
        save_path = os.path.join(save_dir, f"{obj_name}_llm_{level}.json")
    else:
        save_path = os.path.join(save_dir, f"{obj_name}_{level}.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(info_dict, f, ensure_ascii=False, indent=2)

def save_list_info(list_obj, filename, save_dir="../../reports/phase_1_4/lists_info/librespeech", is_llm=False):
    """
    Save a list of dictionaries to a JSON file.

    Parameters
    ----------
    list_obj : list[dict]
        The list of dictionaries to save.
    filename : str
        Base name for the saved file (without extension).
    save_dir : str, optional
        Directory where the file will be saved.
    """
    os.makedirs(save_dir, exist_ok=True)

    if is_llm:
        save_path = os.path.join(save_dir, f"{filename}_llm_.json")
    else:
        save_path = os.path.join(save_dir, f"{filename}.json")

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(list_obj, f, ensure_ascii=False, indent=2)

## Cohere

In [5]:
from stt_benchmarking.models.speech_to_text.english import cohere_english
importlib.reload(cohere_english)

cohere = cohere_english.CohereEnglishInference(device=device)
cohere.run_inference_one_by_one(records)
save_list_info(cohere.samples_info, "cohere_samples")
cohere.overall_metrics

Loading weights: 100%|██████████| 2150/2150 [00:04<00:00, 491.72it/s]
2026-07-14 01:53:17 - [stt_benchmarking.models.speech_to_text.english._load_model:43] INFO - Loaded Cohere Transcribe (03-2026)
Processing Records: 100%|██████████| 2620/2620 [11:06<00:00,  3.93it/s]
2026-07-14 02:04:23 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 11 minutes 6 seconds


{'word_error_rate': {'wer (%)': 1.26,
  'word_similarity_ratio (%)': 99.37,
  'word_edit_distance': 668,
  'substitutions': 508,
  'deletions': 75,
  'insertions': 85,
  'hits': 52444,
  'ref_length': 53027,
  'hyp_length': 53037},
 'character_error_rate': {'cer (%)': 0.38,
  'char_similarity_ratio (%)': 99.81,
  'char_edit_distance': 1071,
  'substitutions': 365,
  'deletions': 347,
  'insertions': 359,
  'hits': 279979,
  'ref_length': 280691,
  'hyp_length': 280703}}

## SpeechBrain

In [5]:
from stt_benchmarking.models.speech_to_text.english import speechbrain
importlib.reload(speechbrain)

sb_object = speechbrain.SpeechBrainInference(device=device)

sb_object.run_inference_one_by_one(records)
save_list_info(sb_object.samples_info, "sb_samples")
sb_object.overall_metrics

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\school.RABAH-DESKTOP\.conda\envs\pytorch_gpu\lib\site-packages\speechbrain\integrations\k2_fsa\__init__.py", line 12, in <module>
    import k2  # noqa
ModuleNotFoundError: No module named 'k2'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\school.RABAH-DESKTOP\.conda\envs\pytorch_gpu\lib\site-packages\speechbrain\utils\importutils.py", line 97, in ensure_module
    self.lazy_module = importlib.import_module(self.target)
  File "c:\Users\school.RABAH-DESKTOP\.conda\envs\pytorch_gpu\lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "<frozen importlib._bootstrap>", line 1050, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1027, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1006, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 688, in 

## Whisper

In [5]:
from stt_benchmarking.models.speech_to_text.english import whisper
importlib.reload(whisper)

whisper_v3 = whisper.FasterWhisperInference(device=device, model_version="v3", batch_size=8)
whisper_v3.run_inference_one_by_one(records)
save_list_info(whisper_v3.samples_info, "whisper_v3_samples")
whisper_v3.overall_metrics

2026-07-14 02:40:54 - [stt_benchmarking.models.speech_to_text.english._load_model:50] INFO - Loaded FasterWhisper V3 Model
2026-07-14 02:40:54 - [stt_benchmarking.models.speech_to_text.english._load_pipeline:61] INFO - Loaded Whisper V3 Pipeline
Processing Records: 100%|██████████| 2620/2620 [39:41<00:00,  1.10it/s]  
2026-07-14 03:20:36 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 39 minutes 41 seconds


{'word_error_rate': {'wer (%)': 2.63,
  'word_similarity_ratio (%)': 98.69,
  'word_edit_distance': 1393,
  'substitutions': 697,
  'deletions': 131,
  'insertions': 565,
  'hits': 52199,
  'ref_length': 53027,
  'hyp_length': 53461},
 'character_error_rate': {'cer (%)': 1.36,
  'char_similarity_ratio (%)': 99.32,
  'char_edit_distance': 3825,
  'substitutions': 535,
  'deletions': 595,
  'insertions': 2695,
  'hits': 279561,
  'ref_length': 280691,
  'hyp_length': 282791}}

## Nvidia

In [5]:
# from stt_benchmarking.models.speech_to_text.english import nvidia
# importlib.reload(nvidia)

# CANARY_OBJ = nvidia.NvidiaInference(device=device, model_id="canary")
# CANARY_OBJ.run_inference_one_by_one(records)
# save_list_info(CANARY_OBJ.samples_info, "canary_samples")
# CANARY_OBJ.overall_metrics
# CANARY_OBJ.reset()

[NeMo I 2026-07-14 05:29:41 mixins:181] Tokenizer CanaryBPETokenizer initialized with 16384 tokens


[NeMo W 2026-07-14 05:29:41 modelPT:180] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 4
    pin_memory: true
    prompt_format: canary2
    max_duration: 40.0
    min_duration: 0.01
    text_field: answer
    lang_field: target_lang
    use_bucketing: true
    max_tps: null
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: null
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-07-14 05:29:41 modelPT:187] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation

[NeMo I 2026-07-14 05:29:41 features:305] PADDING: 0
[NeMo I 2026-07-14 05:29:47 save_restore_connector:275] Model EncDecMultiTaskModel was successfully restored from C:\Users\school.RABAH-DESKTOP\.cache\huggingface\hub\models--nvidia--canary-1b-v2\snapshots\87bc52657add533cd0156b3fc1aef027280754bf\canary-1b-v2.nemo.


2026-07-14 05:29:47 - [stt_benchmarking.models.speech_to_text.english._load_model:53] INFO - Loaded model nvidia canary
Processing Records:   0%|          | 0/2620 [00:00<?, ?it/s][NeMo W 2026-07-14 05:29:47 dataloader:732] The following configuration keys are ignored by Lhotse dataloader: trim_silence
[NeMo W 2026-07-14 05:29:47 dataloader:479] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Processing Records:   0%|          | 1/2620 [00:00<35:12,  1.24it/s][NeMo W 2026-07-14 05:29:47 dataloader:732] The following configuration keys are ignored by Lhotse dataloader: trim_silence
[NeMo W 2026-07-14 05:29:47 dataloader:479] You are using a non-tarr

In [ ]:
# from stt_benchmarking.models.speech_to_text.english import nvidia
# importlib.reload(nvidia)

# CONFORMER_OBJ = nvidia.NvidiaInference(device=device, model_id="conformer")
# CONFORMER_OBJ.run_inference_one_by_one(records)
# save_list_info(CONFORMER_OBJ.samples_info, "conformer_samples")
# CONFORMER_OBJ.overall_metrics

[NeMo I 2026-07-13 19:44:05 mixins:181] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-07-13 19:44:05 modelPT:180] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/NeMo_ASR_SET/English/v2.0/train/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 32
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data/NeMo_ASR_SET/English/v2.0/train/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-07-13 19:44:05 modelPT:187] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/ASR/LibriSpeech/librispeech_withs

[NeMo I 2026-07-13 19:44:05 features:305] PADDING: 0
[NeMo I 2026-07-13 19:44:06 save_restore_connector:275] Model EncDecCTCModelBPE was successfully restored from C:\Users\school.RABAH-DESKTOP\.cache\huggingface\hub\models--nvidia--stt_en_conformer_ctc_large\snapshots\5f18b90411b9fdfc9b52b8caf636f204491afb01\stt_en_conformer_ctc_large.nemo.


2026-07-13 19:44:06 - [stt_benchmarking.models.speech_to_text.english._load_model:50] INFO - Loaded model nvidia conformer
Processing Records: 100%|██████████| 2620/2620 [02:13<00:00, 19.62it/s]
2026-07-13 19:46:20 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 2 minutes 14 seconds


NameError: name 'save_list_info' is not defined

In [6]:
from stt_benchmarking.models.speech_to_text.english import nvidia
importlib.reload(nvidia)

PARAKEET_OBJ = nvidia.NvidiaInference(device=device, model_id="parakeet")
PARAKEET_OBJ.run_inference_one_by_one(records)
save_list_info(PARAKEET_OBJ.samples_info, "parakeet_samples")
PARAKEET_OBJ.overall_metrics
PARAKEET_OBJ.reset()

[NeMo I 2026-07-14 05:42:54 mixins:181] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-07-14 05:42:54 modelPT:180] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-07-14 05:42:54 modelPT:187] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    use_

[NeMo I 2026-07-14 05:42:54 features:305] PADDING: 0
[NeMo I 2026-07-14 05:42:56 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-07-14 05:42:56 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-07-14 05:42:56 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-07-14 05:42:58 save_restore_connector:275] Model EncDecRNNTBPEModel was successfully restored from C:\Users\school.RABAH-DESKTOP\.cache\huggingface\hub\models--nvidia--parakeet-tdt-0.6b-v2\snapshots\ae9ad07059c7c739ffaf932226a8fe64ae2620b0\parakeet-tdt-0.6b-v2.nemo.


2026-07-14 05:42:58 - [stt_benchmarking.models.speech_to_text.english._load_model:53] INFO - Loaded model nvidia parakeet
Processing Records: 100%|██████████| 2620/2620 [02:17<00:00, 19.08it/s]
2026-07-14 05:45:15 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 2 minutes 17 seconds
2026-07-14 05:45:15 - [stt_benchmarking.models.speech_to_text.english.reset:184] INFO - Fastconformer_hybridInference instance has been reset. Model remains loaded.


## Seamless M4t v2

In [6]:
from stt_benchmarking.models.speech_to_text.english import seamlessm4t_v2
importlib.reload(seamlessm4t_v2)

seamless_m4t = seamlessm4t_v2.SeamlessM4TFullInterface(device=device)
seamless_m4t.run_inference_optimized(records)
save_list_info(seamless_m4t.samples_info, "seamless_samples")
seamless_m4t.overall_metrics

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-07-14 03:27:40 - [stt_benchmarking.models.speech_to_text.english._load_model:73] INFO - Loaded Seamless M4T V2 Model
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
2026-07-14 03:27:47 - [stt_benchmarking.models.speech_to_text.english._load_pipeline:99] INFO - Loaded Seamless M4T V2 Pipeline (20s chunks / 1s stride)
2026-07-14 03:27:47 - [stt_benchmarking.models.speech_to_text.english.run_inference_optimized:279] INFO - Running one-by-one inference on 2611 short files (<30.0s)
Processing Records: 100%|██████████| 2611/2611 [47:50<00:00,  1.10s/it]  
2026-07-14 04:15:38 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 47 minutes 50 seconds
2026-07-14 04:15:38 - [stt_benchmarking.models.speech_to_text.english.run_inference_optimized:283] INFO - Running batch inference on 9 long files (≥30.0s)


Processing Batches:   0%|          | 0/9 [00:00<?, ? examples/s]

2026-07-14 04:16:57 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_batch_inference: 1 minutes 19 seconds
2026-07-14 04:16:57 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_optimized: 49 minutes 9 seconds


{'word_error_rate': {'wer (%)': 2.69,
  'word_similarity_ratio (%)': 98.66,
  'word_edit_distance': 1427,
  'substitutions': 1084,
  'deletions': 126,
  'insertions': 217,
  'hits': 51817,
  'ref_length': 53027,
  'hyp_length': 53118},
 'character_error_rate': {'cer (%)': 1.0,
  'char_similarity_ratio (%)': 99.5,
  'char_edit_distance': 2798,
  'substitutions': 1039,
  'deletions': 782,
  'insertions': 977,
  'hits': 278870,
  'ref_length': 280691,
  'hyp_length': 280886}}

In [6]:
import os
import json

def load_samples_info(file_name, file_dr="../../results/librespeech/audios"):
    """
    Load a JSON file from the given directory and normalize paths to start with ../../data
    
    Args:
        file_name (str): Name of the JSON file (with .json extension).
        file_dr (str): Directory where the file is stored.

    Returns:
        dict: Parsed JSON data with normalized paths as keys.
    """
    file_path = os.path.join(file_dr, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    normalized_data = {}
    for key, value in data.items():        
        normalized_data[key] = value
    
    return normalized_data

In [7]:
def collect_model_metrics(models_with_names):
    """
    Collect metrics from multiple model samples_info objects into a unified DataFrame.
    
    Parameters
    ----------
    models_with_names : list[tuple[str, dict]]
        List of tuples (model_name, samples_info_dict).
    
    Returns
    -------
    pd.DataFrame
        DataFrame containing all model metrics and metadata.
    """
    records = []
    for model_name, samples_info in models_with_names:
        for audio_id, content in samples_info.items():
            metrics = content["metrics"]
            wer_data = metrics["word_error_rate"]
            cer_data = metrics["character_error_rate"]

            wer = wer_data["wer (%)"]
            cer = cer_data["cer (%)"]

            if model_name in {"ensemble", "hybrid_ensemble", "rover"}:
                inference_time = content.get("fusion_time")
            else:
                inference_time = content.get("inference_time")

            records.append({
                "audio_id": audio_id,
                "model": model_name,
                "raw_transcription": content.get("raw_transcription"),
                "normalized_transcription": content.get("normalized_transcription"),
                "raw_prediction": content.get("raw_prediction"),
                "normalized_prediction": content.get("normalized_prediction"),
                "inference_time": inference_time,
                "rtf": content.get("rtf"),
                "fusion_time": content.get("fusion_time"),
                "duration": content.get("duration"),
                "wer": wer,
                "substitutions_w": wer_data["substitutions"],
                "deletions_w": wer_data["deletions"],
                "insertions_w": wer_data["insertions"],
                "hits_w": wer_data["hits"],
                "cer": cer,
                "substitutions_c": cer_data["substitutions"],
                "deletions_c": cer_data["deletions"],
                "insertions_c": cer_data["insertions"],
                "hits_c": cer_data["hits"],
            })

    return pd.DataFrame(records)

def filter_audio_records(df_samples, ensemble_name="ensemble"):
    empty = df_samples["normalized_prediction"].fillna("").str.strip() == ""
    is_ens = df_samples["model"] == ensemble_name
    ens_empty = (empty & is_ens).groupby(df_samples["audio_id"]).transform("any")

    empty_gt = df_samples["normalized_transcription"].fillna("").str.strip() == ""
    gt_empty = empty_gt.groupby(df_samples["audio_id"]).transform("any")

    drop = ens_empty | gt_empty
    df_filtered = df_samples[~drop].reset_index(drop=True)

    print(f"Audios dropped (empty ground truth): {df_samples.loc[gt_empty, 'audio_id'].nunique()}")
    print(f"Audios dropped (empty {ensemble_name}): {df_samples.loc[ens_empty, 'audio_id'].nunique()}")
    print(f"Audios: {df_samples['audio_id'].nunique()} -> {df_filtered['audio_id'].nunique()}")
    return df_filtered

def evaluate_absolute_metrics(df_filtered):
    """
    Compute absolute WER/CER metrics for each model against human references.
    """
    agg_results = []
    for model, g in df_filtered.groupby("model"):
        refs = g["normalized_transcription"].tolist()
        hyps = g["normalized_prediction"].tolist()

        duration_mean = pd.Series(g.get("duration", [])).dropna().mean()
        inference_time_mean = pd.Series(g.get("inference_time", [])).dropna().mean()
        rtf_mean = pd.Series(g.get("rtf", [])).dropna().mean()
        fusion_time_mean = pd.Series(g.get("fusion_time", [])).dropna().mean()
        eval_result = mt.BasicSTTMetrics.evaluate(refs=refs, hyps=hyps)
        word_metrics = eval_result['word_error_rate']
        char_metrics = eval_result['character_error_rate']

        agg_results.append({
            "model": model,
            "WER (%)": word_metrics["wer (%)"],
            "w_substitutions": word_metrics["substitutions"],
            "w_deletions": word_metrics["deletions"],
            "w_insertions": word_metrics["insertions"],
            "w_hits": word_metrics["hits"],
            "word_edit_distance": word_metrics["word_edit_distance"],
            "word_similarity_ratio (%)": word_metrics["word_similarity_ratio (%)"],
            "w_ref_length": word_metrics["ref_length"],
            "w_hyp_length": word_metrics["hyp_length"],

            "CER (%)": char_metrics["cer (%)"],
            "c_substitutions": char_metrics["substitutions"],
            "c_deletions": char_metrics["deletions"],
            "c_insertions": char_metrics["insertions"],
            "c_hits": char_metrics["hits"],
            "char_edit_distance": char_metrics["char_edit_distance"],
            "char_similarity_ratio (%)": char_metrics["char_similarity_ratio (%)"],
            "c_ref_length": char_metrics["ref_length"],
            "c_hyp_length": char_metrics["hyp_length"],

            "duration_mean": duration_mean if not pd.isna(duration_mean) else None,
            "inference_time_mean": inference_time_mean if not pd.isna(inference_time_mean) else None,
            "rtf_mean": rtf_mean if not pd.isna(rtf_mean) else None,
            "inference_time_total_h": (inference_time_mean * len(g) / 3600) if not pd.isna(inference_time_mean) else None,
            "fusion_time_mean": fusion_time_mean
            
        })

    df_eval = pd.DataFrame(agg_results)
    df_eval = df_eval.sort_values(
        by=["WER (%)", "word_similarity_ratio (%)"],
        ascending=[True, False]
    ).reset_index(drop=True)
    return df_eval

def evaluate_relative_metrics(df_filtered, model_name, exclude_models=[]):
    """
    Compute relative WER/CER metrics for each model using ensemble outputs as reference.
    """
    agg_results = []

    ensemble_df = (
        df_filtered[df_filtered['model'] == model_name]
        [["audio_id", "normalized_prediction"]]
        .reset_index(drop=True)
    )
    ensemble_df['_order'] = range(len(ensemble_df))

    assert ensemble_df["audio_id"].is_unique, "Duplicate audio_id in ensemble."
    assert ensemble_df["normalized_prediction"].notna().all(), "Ensemble contains None normalized_prediction."

    models_to_compare = df_filtered[~df_filtered['model'].isin(exclude_models)]

    for model, g in models_to_compare.groupby("model"):
        g = g[["audio_id", "normalized_prediction"]].reset_index(drop=True)

        assert g["audio_id"].is_unique, f"Duplicate audio_id in model {model}."
        assert g["normalized_prediction"].notna().all(), f"Model {model} contains None normalized_prediction."

        merged = ensemble_df.merge(g, on="audio_id", suffixes=("_ref", "_hyp"), how="inner")
        merged = merged.sort_values("_order").reset_index(drop=True)
        assert len(merged) == len(ensemble_df), f"Alignment mismatch for {model}: {len(merged)} vs {len(ensemble_df)}."

        refs = merged["normalized_prediction_ref"].tolist()
        hyps = merged["normalized_prediction_hyp"].tolist()
        assert len(refs) > 0, f"No valid reference for model {model}."

        eval_result = mt.BasicSTTMetrics.evaluate(refs=refs, hyps=hyps)
        word_metrics = eval_result['word_error_rate']
        char_metrics = eval_result['character_error_rate']

        agg_results.append({
            "model": model,
            "WER (%)": word_metrics["wer (%)"],
            "w_substitutions": word_metrics["substitutions"],
            "w_deletions": word_metrics["deletions"],
            "w_insertions": word_metrics["insertions"],
            "w_hits": word_metrics["hits"],
            "word_edit_distance": word_metrics["word_edit_distance"],
            "word_similarity_ratio (%)": word_metrics["word_similarity_ratio (%)"],
            "w_ref_length": word_metrics["ref_length"],
            "w_hyp_length": word_metrics["hyp_length"],

            "CER (%)": char_metrics["cer (%)"],
            "c_substitutions": char_metrics["substitutions"],
            "c_deletions": char_metrics["deletions"],
            "c_insertions": char_metrics["insertions"],
            "c_hits": char_metrics["hits"],
            "char_edit_distance": char_metrics["char_edit_distance"],
            "char_similarity_ratio (%)": char_metrics["char_similarity_ratio (%)"],
            "c_ref_length": char_metrics["ref_length"],
            "c_hyp_length": char_metrics["hyp_length"],
        })

    df_relative = pd.DataFrame(agg_results)
    df_relative = df_relative.sort_values(
        by=["WER (%)", "word_similarity_ratio (%)"],
        ascending=[True, False]
    ).reset_index(drop=True)
    return df_relative

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau, pearsonr

# assumes `mt` (your metrics module) is importable in the calling scope,
# i.e. mt.BasicSTTMetrics.evaluate(refs=..., hyps=...) as used elsewhere.


def add_pseudo_reference_wer(df_filtered, ensemble, transcript_key="normalized_prediction"):
    """
    Add per-sample pseudo-reference WER/CER.

    ensemble : dict[audio_id -> dict] with the fused transcript under `transcript_key`.
    """
    ens = {aid: v[transcript_key] for aid, v in ensemble.items()}

    refs = df_filtered["audio_id"].map(ens)
    hyps = df_filtered[transcript_key]

    pseudo_wer, pseudo_cer = [], []
    for ref, hyp in zip(refs, hyps):
        if not isinstance(ref, str) or not ref.strip() or not isinstance(hyp, str):
            pseudo_wer.append(np.nan)
            pseudo_cer.append(np.nan)
            continue
        r = mt.BasicSTTMetrics.evaluate(refs=ref, hyps=hyp)
        pseudo_wer.append(r["word_error_rate"]["wer (%)"])
        pseudo_cer.append(r["character_error_rate"]["cer (%)"])

    df = df_filtered.copy()
    df["pseudo_wer"] = pseudo_wer
    df["pseudo_cer"] = pseudo_cer

    missing = refs.isna().sum()
    if missing:
        print(f"Rows with no ensemble reference: {missing}")
    return df

def _corr(x, y):
    if len(x) < 3 or pd.Series(x).nunique() < 2 or pd.Series(y).nunique() < 2:
        return None
    sp = spearmanr(x, y)
    kt = kendalltau(x, y)
    pr = pearsonr(x, y)
    return {
        "pearson": pr.statistic, "pearson_p": pr.pvalue,
        "spearman": sp.statistic, "spearman_p": sp.pvalue,
        "kendall": kt.statistic, "kendall_p": kt.pvalue,
    }


def per_sample_correlation(df_pseudo, exclude_models=("ensemble", "rover"),
                           metric_pair=("wer", "pseudo_wer"), min_pairs=30):
    """
    Option A. Per-sample WER tracking: correlate true WER vs pseudo-reference WER.

    Returns a df with one row per model (within-model, across samples) plus an
    'ALL' row (pooled over every sample-model pair, the direct NoRefER analogue).
    """
    true_col, pseudo_col = metric_pair
    sub = df_pseudo[~df_pseudo["model"].isin(exclude_models)].copy()
    rows = []

    for model, g in sub.groupby("model"):
        d = g[[true_col, pseudo_col]].dropna()
        if len(d) < min_pairs:
            continue
        c = _corr(d[true_col].values, d[pseudo_col].values)
        if c:
            rows.append({"model": model, "n": len(d), **c})

    d_all = sub[[true_col, pseudo_col]].dropna()
    c_all = _corr(d_all[true_col].values, d_all[pseudo_col].values)
    if c_all:
        rows.append({"model": "ALL (pooled)", "n": len(d_all), **c_all})

    return pd.DataFrame(rows).sort_values("spearman", ascending=False).reset_index(drop=True)


def per_sample_ranking(df_pseudo, exclude_models=("ensemble", "rover"),
                       metric_cols=("wer", "pseudo_wer"),
                       tiebreak_cols=("cer", "pseudo_cer"),
                       min_models=3):
    """
    Per-sample ranking agreement, WER primary with CER as tiebreak only.

    For each sample, systems are ordered by true (WER, then CER) and by
    pseudo (WER, then CER); CER is consulted solely to break exact WER ties,
    so the axis remains a WER ranking, not a WER+CER blend. The two orderings
    are correlated within the sample, then aggregated across samples.

    Returns a one-row summary df with mean/median/std of Spearman and Kendall,
    usable/dropped sample counts, and the residual WER tie fraction.
    """
    true_w, pseudo_w = metric_cols
    true_c, pseudo_c = tiebreak_cols
    sub = df_pseudo[~df_pseudo["model"].isin(exclude_models)].copy()

    sp_vals, kt_vals, tie_fracs = [], [], []
    n_dropped = 0

    for audio_id, g in sub.groupby("audio_id"):
        d = g[[true_w, pseudo_w, true_c, pseudo_c]].dropna()
        if len(d) < min_models:
            n_dropped += 1
            continue

        # residual WER-only tie fraction, before CER breaks them (diagnostic)
        _, counts = np.unique(d[true_w].values, return_counts=True)
        tie_fracs.append(1.0 - len(counts) / len(d))

        # composite ranks: WER primary, CER tiebreak. lexsort keys are
        # applied last-key-first, so primary key goes last.
        true_rank = np.lexsort((d[true_c].values, d[true_w].values)).argsort()
        pseudo_rank = np.lexsort((d[pseudo_c].values, d[pseudo_w].values)).argsort()

        if pd.Series(true_rank).nunique() < 2 or pd.Series(pseudo_rank).nunique() < 2:
            n_dropped += 1
            continue

        sp_vals.append(spearmanr(true_rank, pseudo_rank).statistic)
        kt_vals.append(kendalltau(true_rank, pseudo_rank).statistic)

    summary = {
        "n_samples_used": len(sp_vals),
        "n_samples_dropped": n_dropped,
        "mean_wer_ties": float(np.mean(tie_fracs)) if tie_fracs else np.nan,
        "spearman_mean": float(np.nanmean(sp_vals)) if sp_vals else np.nan,
        "spearman_median": float(np.nanmedian(sp_vals)) if sp_vals else np.nan,
        "spearman_std": float(np.nanstd(sp_vals)) if sp_vals else np.nan,
        "kendall_mean": float(np.nanmean(kt_vals)) if kt_vals else np.nan,
        "kendall_median": float(np.nanmedian(kt_vals)) if kt_vals else np.nan,
        "kendall_std": float(np.nanstd(kt_vals)) if kt_vals else np.nan,
    }
    return pd.DataFrame([summary])


def ranking_correlation(df_absolute, df_relative, exclude_models=("ensemble", "rover")):
    """
    Option B (headline). Correlate the reference ranking against the
    pseudo-reference ranking over models, on full-set aggregate WER.

    df_absolute : output of evaluate_absolute_metrics
    df_relative : output of evaluate_relative_metrics
    Returns (merged_df, coefficients_dict).
    """
    a = df_absolute[["model", "WER (%)"]].rename(columns={"WER (%)": "wer_abs"})
    r = df_relative[["model", "WER (%)"]].rename(columns={"WER (%)": "wer_rel"})
    m = a.merge(r, on="model", how="inner")
    m = m[~m["model"].isin(exclude_models)].reset_index(drop=True)

    c = _corr(m["wer_abs"].values, m["wer_rel"].values)
    return m, c

In [9]:
seamless_samples = load_samples_info("seamless_samples.json")
whisper_samples = load_samples_info("whisper_v3_samples.json")
canary_samples = load_samples_info("canary_samples.json")
conformer_samples = load_samples_info("conformer_samples.json")
parakeet_samples = load_samples_info("parakeet_samples.json")
sb_samples = load_samples_info("sb_samples_flat.json")
cohere_samples = load_samples_info("cohere_samples.json")

In [10]:
def norm(p):
    return os.path.normpath(p).replace("\\","/")

exclude_paths={norm(r["audio_path"]) for r in records_bigger_30}

def drop_excluded(samples):
    return {k:v for k,v in samples.items() if norm(k) not in exclude_paths}

seamless_samples=drop_excluded(seamless_samples)
whisper_samples=drop_excluded(whisper_samples)
canary_samples=drop_excluded(canary_samples)
conformer_samples=drop_excluded(conformer_samples)
parakeet_samples=drop_excluded(parakeet_samples)
sb_samples=drop_excluded(sb_samples)
cohere_samples=drop_excluded(cohere_samples)

In [11]:
all_models = [
    seamless_samples,
    whisper_samples,
    canary_samples,
    conformer_samples,
    parakeet_samples,
    sb_samples,
    cohere_samples
]

model_names = [
    "seamless_m4t",
    "whisper_v3",
    "canary",
    "conformer",
    "parakeet",
    "sb",
    "cohere"
]
voters = ["seamless_m4t", "whisper_v3", "parakeet", "sb", "cohere"]
voters_as_elements = [seamless_samples, whisper_samples, parakeet_samples, sb_samples, cohere_samples]

## ROVER Evaluation Protocol

In [12]:
from stt_benchmarking.utils import evaluation_protocol
from stt_benchmarking.models.speech_to_text.english import ensemble, rover
importlib.reload(evaluation_protocol)

proto = evaluation_protocol.EvaluationProtocol(
    model_samples=dict(zip(model_names, all_models)),
    voters=voters,
    ensemble_factory=rover.ROVER,
    outdir="protocol_out/librispeech_clean_final_rover1.1",
    n_boot=10000, seed=42, min_pool_size=3,
    suffix="_rover_librespeech",
)
report = proto.run()

2026-09-09 13:50:42 - [stt_benchmarking.utils.evaluation_protocol.step0_decorrelation:265] INFO - Step 0 Finished.
ROVER Fusion...: 100%|██████████| 2611/2611 [00:06<00:00, 400.67it/s]
2026-09-09 13:51:21 - [stt_benchmarking.utils.evaluation_protocol.step1_bias:295] INFO - Step 1 Finished.
2026-09-09 13:51:21 - [stt_benchmarking.utils.evaluation_protocol.step2_ranking:328] INFO - Step 2 Finished.
ROVER Fusion...: 100%|██████████| 2611/2611 [00:08<00:00, 290.18it/s]
2026-09-09 13:51:33 - [stt_benchmarking.utils.evaluation_protocol.step3_pipeline:383] INFO - Step 3 Finished.
2026-09-09 13:51:35 - [stt_benchmarking.utils.evaluation_protocol.step4_bootstrap:427] INFO - Step 4 Finished.
2026-09-09 13:51:37 - [stt_benchmarking.utils.evaluation_protocol.step4_bootstrap:427] INFO - Step 4 Finished.
2026-09-09 13:51:46 - [stt_benchmarking.utils.evaluation_protocol.step5_baselines:471] INFO - Step 5 Finished.
ROVER Fusion...: 100%|██████████| 2611/2611 [00:01<00:00, 1640.04it/s] 
2026-09-09 13:5


=== skipped_empty_gt ===
0

=== main_kwargs_applied ===
{}

=== main_kwargs_dropped ===
[]

=== step0_cross_wer ===
              seamless_m4t  whisper_v3  canary  conformer  parakeet    sb  cohere
seamless_m4t          0.00        2.84    2.21       2.38      2.14  2.56    2.14
whisper_v3            2.84        0.00    2.35       2.60      2.22  2.81    2.16
canary                2.21        2.35    0.00       1.90      1.27  2.16    1.54
conformer             2.38        2.60    1.90       0.00      1.60  1.89    1.60
parakeet              2.14        2.22    1.27       1.60      0.00  1.88    1.29
sb                    2.56        2.81    2.16       1.89      1.88  0.00    1.74
cohere                2.14        2.16    1.54       1.60      1.29  1.74    0.00

=== step0_err_corr ===
              seamless_m4t  whisper_v3  canary  conformer  parakeet      sb  cohere
seamless_m4t        1.0000      0.5964  0.6161     0.6072    0.6198  0.5441  0.5400
whisper_v3          0.5964      1.0

## Ensemble Evaluation Protocol

In [13]:
from stt_benchmarking.utils import evaluation_protocol
from stt_benchmarking.models.speech_to_text.english import ensemble
importlib.reload(evaluation_protocol)

proto = evaluation_protocol.EvaluationProtocol(
    model_samples=dict(zip(model_names, all_models)),
    voters=voters,
    ensemble_factory=ensemble.HybridEnsemble,
    outdir="protocol_out/librispeech_validated_ensemble",
    n_boot=10000, seed=42, min_pool_size=3,
    suffix="_validated_ensemble_librispeech",
)
report = proto.run()

2026-09-22 17:33:02 - [stt_benchmarking.utils.evaluation_protocol.step0_decorrelation:265] INFO - Step 0 Finished.
Fusing Inputs...: 100%|██████████| 2611/2611 [00:04<00:00, 582.68it/s]
2026-09-22 17:33:31 - [stt_benchmarking.utils.evaluation_protocol.step1_bias:295] INFO - Step 1 Finished.
2026-09-22 17:33:31 - [stt_benchmarking.utils.evaluation_protocol.step2_ranking:328] INFO - Step 2 Finished.
Fusing Inputs...: 100%|██████████| 2611/2611 [00:05<00:00, 477.06it/s]
2026-09-22 17:33:36 - [stt_benchmarking.utils.evaluation_protocol.step3_pipeline:383] INFO - Step 3 Finished.
2026-09-22 17:33:37 - [stt_benchmarking.utils.evaluation_protocol.step4_bootstrap:427] INFO - Step 4 Finished.
2026-09-22 17:33:38 - [stt_benchmarking.utils.evaluation_protocol.step4_bootstrap:427] INFO - Step 4 Finished.
2026-09-22 17:33:42 - [stt_benchmarking.utils.evaluation_protocol.step5_baselines:471] INFO - Step 5 Finished.
Fusing Inputs...: 100%|██████████| 2611/2611 [00:02<00:00, 1176.12it/s]
2026-09-22 17


=== skipped_empty_gt ===
0

=== main_kwargs_applied ===
{}

=== main_kwargs_dropped ===
[]

=== step0_cross_wer ===
              seamless_m4t  whisper_v3  canary  conformer  parakeet    sb  cohere
seamless_m4t          0.00        2.84    2.21       2.38      2.14  2.56    2.14
whisper_v3            2.84        0.00    2.35       2.60      2.22  2.81    2.16
canary                2.21        2.35    0.00       1.90      1.27  2.16    1.54
conformer             2.38        2.60    1.90       0.00      1.60  1.89    1.60
parakeet              2.14        2.22    1.27       1.60      0.00  1.88    1.29
sb                    2.56        2.81    2.16       1.89      1.88  0.00    1.74
cohere                2.14        2.16    1.54       1.60      1.29  1.74    0.00

=== step0_err_corr ===
              seamless_m4t  whisper_v3  canary  conformer  parakeet      sb  cohere
seamless_m4t        1.0000      0.5964  0.6161     0.6072    0.6198  0.5441  0.5400
whisper_v3          0.5964      1.0

In [15]:
from stt_benchmarking.utils import deployment_protocol as dp
from stt_benchmarking.models.speech_to_text.english import ensemble

importlib.reload(dp)

dp = dp.DeploymentProtocol.from_hypotheses(
    model_samples=dict(zip(model_names, all_models)),
    voters=model_names,
    ensemble_factory=ensemble.HybridEnsemble,
    n_boot=1000, seed=42, min_pool_size=3,
    outdir="deployment_out/librispeech_validated_all_voting_ensemble", suffix="librispeech_validated_all_voting_ensemble")
    
r = dp.run(weighting="filtration")

2026-09-22 17:44:44 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function cross_wer: 0 minutes 1 seconds (1.5s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:12<00:00, 213.08it/s]
2026-09-22 17:46:06 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function ranking: 1 minutes 22 seconds (82.3s)
2026-09-22 17:46:06 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function ranking: 0 minutes 0 seconds (0.0s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:14<00:00, 178.65it/s]
2026-09-22 17:54:44 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function model_filtration: 8 minutes 38 seconds (518.4s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:01<00:00, 1624.63it/s]
2026-09-22 17:57:29 - [stt_benchmarking.utils.deployment_protocol.rank_distribution:498] INFO - Rank distribution finished.
2026-09-22 17:57:29 - [stt

In [14]:
from stt_benchmarking.utils import deployment_protocol as dp
from stt_benchmarking.models.speech_to_text.english import ensemble

dp = dp.DeploymentProtocol.from_hypotheses(
    model_samples={n: s for n, s in zip(model_names, all_models) if n in voters},
    voters=voters,
    ensemble_factory=ensemble.HybridEnsemble,
    n_boot=1000, seed=42, min_pool_size=3,
    outdir="deployment_out/librispeech_validated_voters_ensemble", suffix="librispeech_validated_voters_ensemble")

r = dp.run(weighting="filtration")

2026-09-22 17:43:37 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function cross_wer: 0 minutes 0 seconds (0.7s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:04<00:00, 635.05it/s]
2026-09-22 17:44:00 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function ranking: 0 minutes 23 seconds (23.2s)
2026-09-22 17:44:00 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function ranking: 0 minutes 0 seconds (0.0s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:05<00:00, 519.44it/s]
2026-09-22 17:44:06 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function model_filtration: 0 minutes 5 seconds (5.1s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:01<00:00, 1610.98it/s]
2026-09-22 17:44:28 - [stt_benchmarking.utils.deployment_protocol.rank_distribution:498] INFO - Rank distribution finished.
2026-09-22 17:44:28 - [stt_be

### Uniofmr Rank Distribution - Ensemble Deployment Protocol

In [12]:
from stt_benchmarking.utils import deployment_protocol as dp
from stt_benchmarking.models.speech_to_text.english import ensemble

importlib.reload(dp)

dp = dp.DeploymentProtocol.from_hypotheses(
    model_samples={n: s for n, s in zip(model_names, all_models) if n in voters},
    voters=voters,
    ensemble_factory=ensemble.HybridEnsemble,
    n_boot=1000, seed=42, min_pool_size=3,
    outdir="deployment_out/librispeech_validated_voters_unweighted_ensemble", suffix="librispeech_validated_voters_unweighted_ensemble")

rep = dp.run(weighting="uniform")

2026-09-22 22:30:11 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function cross_wer: 0 minutes 1 seconds (1.6s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:08<00:00, 311.31it/s]
2026-09-22 22:30:55 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function ranking: 0 minutes 43 seconds (43.5s)
2026-09-22 22:30:55 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function ranking: 0 minutes 0 seconds (0.0s)
Fusing Inputs...: 100%|██████████| 2611/2611 [00:03<00:00, 869.61it/s]
2026-09-22 22:31:33 - [stt_benchmarking.utils.deployment_protocol.rank_distribution:498] INFO - Rank distribution finished.
2026-09-22 22:31:33 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function rank_distribution: 0 minutes 38 seconds (38.7s)
2026-09-22 22:31:33 - [stt_benchmarking.utils.deployment_protocol.run:509] INFO - Deployment proto

## ROVER

In [12]:
from stt_benchmarking.models.speech_to_text.english import rover
importlib.reload(rover)

ROVEROBJECT = rover.ROVER()
ROVEROBJECT.combine_models_transcriptions(
    seamless_samples,
    whisper_samples,
    parakeet_samples,
    sb_samples,
    cohere_samples
)
ROVEROBJECT.main()
ROVEROBJECT.eval(filtered_records)
rover_samples = ROVEROBJECT.fusion_results
ROVEROBJECT.overall_metrics

ROVER Fusion...: 100%|██████████| 2611/2611 [00:05<00:00, 515.19it/s]


{'word_error_rate': {'wer (%)': 1.38,
  'word_similarity_ratio (%)': 99.31,
  'word_edit_distance': 721,
  'substitutions': 534,
  'deletions': 78,
  'insertions': 109,
  'hits': 51737,
  'ref_length': 52349,
  'hyp_length': 52380},
 'character_error_rate': {'cer (%)': 0.42,
  'char_similarity_ratio (%)': 99.79,
  'char_edit_distance': 1162,
  'substitutions': 377,
  'deletions': 374,
  'insertions': 411,
  'hits': 276242,
  'ref_length': 276993,
  'hyp_length': 277030}}

In [1]:
ROVEROBJECT.fusion_wall_time

NameError: name 'ROVEROBJECT' is not defined

## Ensemble

In [12]:
from stt_benchmarking.models.speech_to_text.english import ensemble
importlib.reload(ensemble)

ENSEMBLE_OBJ = ensemble.HybridEnsemble()
ENSEMBLE_OBJ.combine_models_transcriptions(
    seamless_samples,
    whisper_samples,
    parakeet_samples,
    sb_samples,
    cohere_samples
)
ENSEMBLE_OBJ.main()
ENSEMBLE_OBJ.eval(filtered_records)
ensemble_samples = ENSEMBLE_OBJ.samples_info
ENSEMBLE_OBJ.overall_metrics

Fusing Inputs...: 100%|██████████| 2611/2611 [00:06<00:00, 403.51it/s]


{'word_error_rate': {'wer (%)': 1.37,
  'word_similarity_ratio (%)': 99.32,
  'word_edit_distance': 717,
  'substitutions': 534,
  'deletions': 87,
  'insertions': 96,
  'hits': 51728,
  'ref_length': 52349,
  'hyp_length': 52358},
 'character_error_rate': {'cer (%)': 0.41,
  'char_similarity_ratio (%)': 99.79,
  'char_edit_distance': 1142,
  'substitutions': 370,
  'deletions': 419,
  'insertions': 353,
  'hits': 276204,
  'ref_length': 276993,
  'hyp_length': 276927}}

In [13]:
ENSEMBLE_OBJ.fusion_wall_time

6.596327543258667

In [14]:
model_dicts = {
    "seamless_m4t": seamless_samples,
    "whisper_v3": whisper_samples,
    "parakeet": parakeet_samples,
    "sb": sb_samples,
    "cohere": cohere_samples,
}
disp = {
    "seamless_m4t": "SeamlessM4T v2",
    "whisper_v3": "Whisper large-v3",
    "parakeet": "Parakeet",
    "sb": "SpeechBrain",
    "cohere": "Cohere Transcribe",
}
assert set(model_dicts) == set(voters) == set(disp), (set(model_dicts), set(voters), set(disp))

In [15]:
def wer_of(samples, path):
    m = samples.get(path, {}).get("metrics")
    if not m: return None
    return m["word_error_rate"]["wer (%)"]

In [16]:
rows = []
for path in ensemble_samples:
    ens_wer = wer_of(ensemble_samples, path)
    if ens_wer is None: continue
    sys_wers = {n: wer_of(d, path) for n, d in model_dicts.items()}
    sys_wers = {n: w for n, w in sys_wers.items() if w is not None}
    if not sys_wers: continue
    best_single = min(sys_wers.values())
    margin = best_single - ens_wer
    ref = ensemble_samples[path].get("normalized_transcription", "")
    n_words = len(ref.split())
    md = ensemble_samples[path]["metadata"]
    n_filtered = md["records_filtration"].get("num_filtered", 0)
    n_ties = sum(
        1 for vd in md["voting"]["voting_details"]
        if vd.get("token_votes") and
        sorted(vd["token_votes"].values())[-1] == (sorted(vd["token_votes"].values())[-2] if len(vd["token_votes"])>1 else -1)
    )
    rows.append({"path": path, "ens_wer": round(ens_wer,2), "best_single": round(best_single,2),
                 "margin": round(margin,2), "n_words": n_words, "n_filtered": n_filtered,
                 "n_ties": n_ties, "strategy": md["reference_selection"].get("strategy_metric")})

cand = [r for r in rows if r["margin"] > 0 and 5 <= r["n_words"] <= 12]
cand.sort(key=lambda r: (r["n_filtered"]>0, r["n_ties"]>0, r["margin"]), reverse=True)

for r in cand[:15]: 
    print(r)

In [17]:
print("rows:", len(rows))
if rows:
    import statistics
    print("n_words range:", min(r["n_words"] for r in rows), "-", max(r["n_words"] for r in rows))
    print("margin>0 count:", sum(1 for r in rows if r["margin"] > 0))
    print("n_words 5-12 count:", sum(1 for r in rows if 5 <= r["n_words"] <= 12))
print("cand:", len(cand))

rows: 2611
n_words range: 1 - 87
margin>0 count: 2
n_words 5-12 count: 851
cand: 0


In [18]:
print(ensemble_samples[list(ensemble_samples)[0]]["normalized_prediction"])
print(ensemble_samples[list(ensemble_samples)[0]]["normalized_transcription"])

he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce


In [21]:
cand = sorted(rows, key=lambda r: (-r["margin"], -(r["n_filtered"]>0), -(r["n_ties"]>0)))
cand = [r for r in cand if 5 <= r["n_words"] <= 25][:15]
for r in cand: print(r)

{'path': '../../data/librispeech/LibriSpeech/test-clean\\2094\\142345\\2094-142345-0028.flac', 'ens_wer': 0.0, 'best_single': 6.25, 'margin': 6.25, 'n_words': 16, 'n_filtered': 2, 'n_ties': 1, 'strategy': 'most_common_words'}
{'path': '../../data/librispeech/LibriSpeech/test-clean\\1089\\134691\\1089-134691-0010.flac', 'ens_wer': 60.0, 'best_single': 60.0, 'margin': 0.0, 'n_words': 5, 'n_filtered': 1, 'n_ties': 2, 'strategy': 'most_common_words'}
{'path': '../../data/librispeech/LibriSpeech/test-clean\\1580\\141083\\1580-141083-0018.flac', 'ens_wer': 0.0, 'best_single': 0.0, 'margin': 0.0, 'n_words': 6, 'n_filtered': 1, 'n_ties': 1, 'strategy': 'most_common_words'}
{'path': '../../data/librispeech/LibriSpeech/test-clean\\1995\\1837\\1995-1837-0016.flac', 'ens_wer': 0.0, 'best_single': 0.0, 'margin': 0.0, 'n_words': 17, 'n_filtered': 1, 'n_ties': 1, 'strategy': 'most_common_words'}
{'path': '../../data/librispeech/LibriSpeech/test-clean\\237\\126133\\237-126133-0016.flac', 'ens_wer': 9.

In [27]:
from stt_benchmarking.utils import ensemble_analysis
importlib.reload(ensemble_analysis)

fp=ensemble_analysis.download_amiri("fonts")     # OFL, one-time; returns Amiri-Regular.ttf
AUDIO_ID = "../../data/librispeech/LibriSpeech/test-clean\\2094\\142345\\2094-142345-0028.flac"
ensemble_analysis.render_stages(AUDIO_ID, ensemble_samples, model_dicts, voters, disp,
    ensemble_name="Proposed ensemble", fig_dir="figures_librespeech",
    font_path=fp, max_tokens=16, col_chunk=16)

font: fonts\Amiri-Regular.ttf


TSIV NOT subset; don't know how to subset; dropped


saved: figures_librespeech\stage0_reference.pdf | figures_librespeech\stage0_reference.png


TSIV NOT subset; don't know how to subset; dropped
TSIV NOT subset; don't know how to subset; dropped


saved: figures_librespeech\stage1_filtration.pdf | figures_librespeech\stage1_filtration.png
saved: figures_librespeech\stage2_reference_selection.pdf | figures_librespeech\stage2_reference_selection.png


...

TSIV NOT subset; don't know how to subset; dropped


saved: figures_librespeech\stage4_voting.pdf | figures_librespeech\stage4_voting.png


TSIV NOT subset; don't know how to subset; dropped


saved: figures_librespeech\stage5_comparison.pdf | figures_librespeech\stage5_comparison.png


['figures_librespeech\\stage0_reference.pdf',
 'figures_librespeech\\stage0_reference.png',
 'figures_librespeech\\stage1_filtration.pdf',
 'figures_librespeech\\stage1_filtration.png',
 'figures_librespeech\\stage2_reference_selection.pdf',
 'figures_librespeech\\stage2_reference_selection.png',
 'figures_librespeech\\stage3_alignment.pdf',
 'figures_librespeech\\stage3_alignment.png',
 'figures_librespeech\\stage4_voting.pdf',
 'figures_librespeech\\stage4_voting.png',
 'figures_librespeech\\stage5_comparison.pdf',
 'figures_librespeech\\stage5_comparison.png']

In [14]:
all_models.append(rover_samples)
model_names.append("rover")

all_models.append(ensemble_samples)
model_names.append("ensemble")

In [15]:
models_with_names = list(zip(model_names, all_models))
df_full = collect_model_metrics(models_with_names)
df_filtered_full = filter_audio_records(df_full)

Audios dropped (empty ground truth): 0
Audios dropped (empty ensemble): 0
Audios: 2611 -> 2611


In [16]:
df_full_eval = evaluate_absolute_metrics(df_filtered_full)
df_full_eval[["model", "WER (%)", "CER (%)", "rtf_mean", "inference_time_total_h", "fusion_time_mean"]]

,model,WER (%),CER (%),rtf_mean,inference_time_total_h,fusion_time_mean
0,cohere,1.22,0.37,0.040616,0.182711,NaN
1,ensemble,1.37,0.41,NaN,0.015430,0.021274
2,rover,1.38,0.42,NaN,0.012156,0.016760
3,parakeet,1.67,0.49,0.009393,0.037694,NaN
4,canary,1.89,0.60,0.040443,0.191791,NaN
5,conformer,2.03,0.60,0.009641,0.036378,NaN
6,sb,2.21,0.67,0.176632,1.267476,NaN
7,seamless_m4t,2.57,0.90,0.137370,0.796933,NaN
8,whisper_v3,2.60,1.35,0.138758,0.642775,NaN


In [20]:
df_relative_ensemble = evaluate_relative_metrics(df_filtered_full, "ensemble", exclude_models=["rover", "hybrid_ensemble"])
df_relative_ensemble[["model", "WER (%)", "CER (%)"]]

,model,WER (%),CER (%)
0,ensemble,0.00,0.00
1,cohere,0.75,0.24
2,parakeet,0.83,0.24
3,canary,1.25,0.41
4,conformer,1.32,0.38
5,sb,1.35,0.41
6,seamless_m4t,1.65,0.62
7,whisper_v3,1.70,1.08


# END

In [30]:
# ---- Option B: headline ranking correlation (reference order vs pseudo-reference order) ----
merged_B, coeffs_B = ranking_correlation(df_full_eval, df_relative_ensemble)
print("=== Option B: pooled system-ranking correlation (ensemble as pseudo-reference) ===")
print(merged_B.to_string(index=False))
print(coeffs_B)

# ---- Option A: per-sample pseudo-WER tracking (NoRefER analogue) ----
df_ps = add_pseudo_reference_wer(df_filtered_full, ensemble_samples)
corr_A = per_sample_correlation(df_ps)
corr_A.to_csv("../../reports/phase_1_4/metrics/relative/librespeech/per_sample_correlation_cohere.csv", index=False)
print("\n=== Option A: per-sample WER tracking (per-model + pooled) ===")
print(corr_A.to_string(index=False))

=== Option B: pooled system-ranking correlation (ensemble as pseudo-reference) ===
       model  wer_abs  wer_rel
      cohere     1.26     0.77
    parakeet     1.69     0.83
      canary     1.92     1.27
   conformer     2.06     1.33
          sb     2.23     1.36
  whisper_v3     2.63     1.71
seamless_m4t     2.69     1.77
{'pearson': 0.9707799291244992, 'pearson_p': 0.00027594632496072164, 'spearman': 1.0, 'spearman_p': 0.0, 'kendall': 1.0, 'kendall_p': 0.0003968253968253968}

=== Option A: per-sample WER tracking (per-model + pooled) ===
       model     n  pearson     pearson_p  spearman    spearman_p  kendall     kendall_p
seamless_m4t  2620 0.885930  0.000000e+00  0.754591  0.000000e+00 0.719971  0.000000e+00
   conformer  2620 0.763584  0.000000e+00  0.718475  0.000000e+00 0.685742  0.000000e+00
      canary  2620 0.528124 3.750476e-188  0.673913  0.000000e+00 0.643661 1.283233e-290
          sb  2620 0.575403 6.913579e-231  0.665267  0.000000e+00 0.635745 1.233749e-286
ALL

In [17]:
df_full_rel_rover = evaluate_relative_metrics(df_full, "rover", exclude_models=["ensemble", "rover"])
df_full_rel_ensemble = evaluate_relative_metrics(df_full, "ensemble", exclude_models=["rover", "ensemble"])

In [18]:
df_full_rel_ensemble[["model", "WER (%)", "CER (%)"]]

,model,WER (%),CER (%)
0,parakeet,0.79,0.24
1,cohere,0.82,0.27
2,canary,1.09,0.37
3,conformer,1.20,0.36
4,sb,1.41,0.44
5,whisper_v2,1.80,0.79
6,seamless_m4t,1.82,0.74
7,whisper_v3,1.92,1.25


In [ ]:
df_full_rel_ensemble[["model", "WER (%)", "CER (%)"]]

,model,WER (%),CER (%)
0,parakeet,0.78,0.24
1,canary,1.06,0.37
2,conformer,1.17,0.34
3,sb,1.42,0.43
4,seamless_m4t,1.77,0.71
5,whisper_v2,1.79,0.78
6,whisper_v3,1.92,1.24
